Projeto PI - Entrega (E2) - Análise dos Efeitos
Ana Beatriz Barbosa Yoshida
RA 245609

In [25]:
!unzip -q /dataset.zip -d /content/dados_locais

In [35]:
!unzip -l /dataset.zip | head -n 30

Archive:  /dataset.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        2  2026-04-06 16:34   dataset/.gitignore
        0  2026-04-06 16:29   dataset/175807_futuro/
        0  2026-04-06 16:29   dataset/175807_maria_e_sobel/
        0  2026-04-06 16:29   dataset/175807_pixelular/
        0  2026-04-06 16:29   dataset/186629_canny_edge_detection/
        0  2026-04-06 16:29   dataset/186629_chromatic_aberration_blur/
        0  2026-04-06 16:29   dataset/186629_color_splash/
        0  2026-04-06 16:29   dataset/237310_aberracao_cromatica/
        0  2026-04-06 16:29   dataset/237310_pixelizacao/
        0  2026-04-06 16:29   dataset/237310_quantizacao/
        0  2026-04-06 16:29   dataset/241163_chromatic_aberration/
        0  2026-04-06 16:29   dataset/241163_edge_detection/
        0  2026-04-06 16:29   dataset/241163_pixelation/
        0  2026-04-06 16:29   dataset/243360_chromatic_aberration/
        0  2026-04-06 16:29   dataset/243360_edge_detecti

Importando as bibliotecas e verificação da CPU:

In [26]:
#from google.colab import drive
#drive.mount('/content/drive')

In [27]:
import torch
import torchvision
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Criação do Dataset e Dataloader: padronizar o tamanho das imagens

In [33]:
transformacoes = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])
caminho_dados = '/content/dados_locais/dataset/photos'
dataset = datasets.ImageFolder(root=caminho_dados, transform=transformacoes)

print(f"Total de fotos no dataset: {len(dataset)}")
print(f"Efeitos encontrados (Classes): {dataset.classes}")

dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

FileNotFoundError: [Errno 2] No such file or directory: '/content/dados_locais/dataset/photos'

In [ ]:
def mostrar_lote(imagens, labels, classes):
    fig, axes = plt.subplots(1, 4, figsize=(15, 5))
    for i in range(4):
        img = np.transpose(imagens[i].numpy(), (1, 2, 0))
        axes[i].imshow(img)
        axes[i].set_title(f"Efeito: {classes[labels[i]]}")
        axes[i].axis('off')
    plt.show()

#Pega um lote de imagens do dataloader e mostra
imagens_batch, labels_batch = next(iter(dataloader))
mostrar_lote(imagens_batch, labels_batch, dataset.classes)

Análise não supervisionada com PCA:


In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

todas_imagens = []
todos_labels = []
fotos_processadas = 0
max_fotos = 20

for imagens, labels in dataloader:
    if fotos_processadas >= max_fotos:
        break

    batch_achatado = imagens.view(imagens.size(0), -1).numpy()

    todas_imagens.append(batch_achatado)
    todos_labels.extend(labels.numpy())

    fotos_processadas += len(imagens)


X = np.vstack(todas_imagens)
y = np.array(todos_labels)

print(f"Fotos analisadas: {X.shape[0]}. Colunas (Pixels): {X.shape[1]}")

# 2. Aplicamos o PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='tab10', alpha=0.7, edgecolors='w', s=80)

plt.legend(handles=scatter.legend_elements()[0], labels=dataset.classes, title="Efeitos")
plt.title(f"Análise Não Supervisionada: Agrupamento de Efeitos (Amostra de {max_fotos} fotos)")
plt.xlabel("Componente Principal 1")
plt.ylabel("Componente Principal 2")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()